In [ ]:
import os
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm.auto import tqdm
import re
import joblib

In [ ]:
MODEL_PATH = "/kaggle/input/byt5-base-big-data2" #+ not different preprocess

In [ ]:
TEST_DATA_PATH = "/kaggle/input/deep-past-initiative-machine-translation/test.csv"
BATCH_SIZE = 8 #4
MAX_LENGTH = 512 #512
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH).to(DEVICE)
model.eval()

test_df = pd.read_csv(TEST_DATA_PATH)

In [ ]:
PREFIX = "translate Akkadian to English: "

class InferenceDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.texts = df['transliteration'].astype(str).tolist()
        self.texts = [PREFIX + i for i in self.texts]
        self.tokenizer = tokenizer
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        inputs = self.tokenizer(
            text, 
            max_length=MAX_LENGTH, 
            padding="max_length", 
            truncation=True, 
            return_tensors="pt"
        )
        return {
            "input_ids": inputs["input_ids"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0)
        }

test_dataset = InferenceDataset(test_df, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Starting Inference...")
all_predictions = []

In [ ]:
import torch
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(DEVICE)
model.eval()
model.float()  # <-- force les poids en FP32

all_predictions = []

# Optionnel: si tu veux être sûr de ne pas avoir de comportements bizarres "mixed"
# (sur P100 TF32 n'existe pas, mais ça ne coûte rien)
torch.set_grad_enabled(False)

with torch.inference_mode():  # mieux que no_grad pour l'inférence
    for batch in tqdm(test_loader):
        input_ids = batch["input_ids"].to(DEVICE)              # int64, normal
        attention_mask = batch["attention_mask"].to(DEVICE)    # int64/bool, normal

        # IMPORTANT: ne pas mettre autocast ici (pas de with torch.cuda.amp.autocast())
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=MAX_LENGTH,
            num_beams=4,
            early_stopping=True,
        )

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        all_predictions.extend([d.strip() for d in decoded])

# all_predictions contient une prédiction par élément du test_loader
        predictions = []
        for txt in all_predictions:
            predictions.append(txt)

In [ ]:
print(next(model.parameters()).dtype)  # doit afficher: torch.float32

In [ ]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "translation": predictions
})

submission["translation"] = submission["translation"].apply(lambda x: x if len(x) > 0 else "broken text")

    
submission.to_csv("submission.csv", index=False)
print("Submission file saved successfully!")
submission.head()